# Ensemble learning with scikit-learn

Machine Learning, S8 ELC D1, École Centrale de Lyon, 2027. Yohann De Castro.

**Goal of the session.** We want to decide if an e-mail is a spam. We compare one decision tree, bagging, random
forests and gradient boosting on this task. For each method you will see when it overfits, which hyper-parameters
control it, and how to choose them without the test set. Part 6 ends with a protocol that you can copy for your
project.

| Part | Content | Minutes |
| --- | --- | --- |
| 0 | Setup, data, splits, baseline | 15 |
| 1 | One decision tree: depth and overfitting | 35 |
| 2 | Bootstrap and bagging by hand | 30 |
| 3 | Random forest: out-of-bag error, number of trees, `max_features` | 40 |
| | Break | 15 |
| 4 | Gradient boosting: learning rate, number of trees, early stopping | 50 |
| 5 | Data leakage and `Pipeline` | 20 |
| 6 | Comparison, test set, checklist for the project | 25 |

**How to work.** Run the cells in order. A code cell with a line `# Answer k.m` is for you: replace the line
`raise NotImplementedError(...)` by your code. The questions are in grey boxes, and you write your answer in the cell
below each question. Code cells and questions are numbered separately. Each part uses only the data of Part 0 and
its own variables, so a part that you skip breaks only its own cells (Part 6 tells you what to do in that case).

The free CPU runtime of Google Colab is enough: no GPU is needed. If you lose the session, run all the cells again:
it takes a few minutes.

**Notation.** As in the lectures: $n$ observations, $d$ variables, a fitted classifier $\hat f$. The error of
$\hat f$ on $N$ labelled points $(x_i, y_i)$ is the fraction of misclassified points,
$\frac1N \sum_{i=1}^N \mathbf 1\{\hat f(x_i) \neq y_i\}$. On the training set it is the empirical risk $R_n(\hat f)$
for the 0/1 loss. In scikit-learn, `model.score(X, y)` is the accuracy, so the error is `1 - model.score(X, y)`.

## Part 0. Setup, data, splits, baseline (15 min)

In [ ]:
import sys
import time
import warnings
from urllib.parse import unquote

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (BaggingClassifier, GradientBoostingClassifier, GradientBoostingRegressor,
                              HistGradientBoostingClassifier, RandomForestClassifier)
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

print("Python", sys.version.split()[0])
for module in (np, pd, matplotlib, sklearn):
    print(module.__name__, module.__version__)

SEED = 0      # every random draw of this notebook depends on this seed
N_JOBS = -1   # forests and bagging use all the processors (2 on Colab)
rng = np.random.default_rng(SEED)

**The data.** Spambase (UCI Machine Learning Repository, Hewlett-Packard Labs, 1999) has $n = 4601$ e-mails. Each
e-mail is described by $d = 57$ variables:

- 48 word frequencies: `word_freq_free` is the percentage of the words of the e-mail equal to "free";
- 6 character frequencies, for example `char_freq_(` for the opening parenthesis;
- 3 statistics on the runs of capital letters: average length, longest run, total number of capital letters.

The label is 1 for a spam and 0 otherwise. The cell below reads the copy of the course page, and downloads the data
from OpenML if this copy cannot be read. Both give the same arrays.

In [ ]:
DATA_URL = "https://machine-learning-centrale.github.io/data/spambase.csv"
try:
    df = pd.read_csv(DATA_URL)
    print("Data read from the course page.")
except Exception:
    openml = fetch_openml("spambase", version=1, as_frame=True)
    df = openml.data.copy()
    df.columns = [unquote(name) for name in df.columns]   # OpenML encodes some characters, e.g. %28 for (
    df["spam"] = (openml.target.astype(str) == "1").astype(int)
    print("Data downloaded from OpenML.")

feature_names = list(df.columns[:-1])
X = df[feature_names].to_numpy(dtype=float)
y = df["spam"].to_numpy(dtype=int)
print("X:", X.shape, " y:", y.shape, " missing values:", int(np.isnan(X).sum()))
df[["word_freq_free", "word_freq_remove", "char_freq_(", "capital_run_length_average", "capital_run_length_total"]].describe()

From now on we work with the NumPy arrays `X` (one row per e-mail) and `y`. The function below gives the error of a
fitted classifier.

In [ ]:
def error(model, X, y):
    # Misclassification error of a fitted classifier: 1 - accuracy.
    return 1 - model.score(X, y)

Compute the proportion of spam in the data. You should find a value close to 40 %.

In [ ]:
# Answer 0.1: proportion of spam, one line
raise NotImplementedError("Answer 0.1")
print(f"proportion of spam: {spam_fraction:.3f}")

**Three sets.** Put 20 % of the data aside as the test set (`X_test`, `y_test`). Split the rest (`X_trainval`,
`y_trainval`) into a training set (75 %) and a validation set (25 %). Use `train_test_split` with `stratify` and
`random_state=SEED`. You should find 2760, 920 and 921 rows. The test set is used once, in Part 6.

In [ ]:
# Answer 0.2: two calls to train_test_split, with stratify and random_state=SEED
raise NotImplementedError("Answer 0.2")
print("training", X_train.shape, " validation", X_val.shape, " test", X_test.shape)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)   # the 5 folds of every cross-validation below

**Baseline.** `DummyClassifier(strategy="most_frequent")` predicts the most frequent class of the training set for
every e-mail. Fit it on the training set and compute its validation error.

In [ ]:
# Answer 0.3: fit the constant classifier on the training set, error on the validation set
raise NotImplementedError("Answer 0.3")
print(f"validation error of the constant classifier: {baseline_val_error:.3f}")

> **Question 0.1.** Why do we put the test set aside before we look at any model?

**Your answer.**


> **Question 0.2.** Why do we stratify the splits?

**Your answer.**


> **Question 0.3.** Which error must every model of this session beat?

**Your answer.**


## Part 1. One decision tree: depth and overfitting (35 min)

### 1a. A tree without depth limit

By default `DecisionTreeClassifier` splits the nodes until the leaves are pure. Compute its training and validation
errors, its depth (`get_depth()`) and its number of leaves (`get_n_leaves()`). You should find a training error
close to 0 and a validation error between 8 and 10 %.

In [ ]:
tree = DecisionTreeClassifier(random_state=SEED).fit(X_train, y_train)
# Answer 1.1: training error, validation error, depth and number of leaves
raise NotImplementedError("Answer 1.1")

### 1b. Validation curve of the depth

For `max_depth` from 1 to 25, fit a tree on the training set and store its training and validation errors.

In [ ]:
depths = list(range(1, 26))
train_err, val_err = [], []
for depth in depths:
    # Answer 1.2: fit a tree with this max_depth on the training set; append its two errors
    raise NotImplementedError("Answer 1.2")
best_depth_val = depths[int(np.argmin(val_err))]
print(f"best depth on the validation set: {best_depth_val}, validation error {min(val_err):.3f}")

plt.figure(figsize=(7, 4))
plt.plot(depths, train_err, "o-", label="training error")
plt.plot(depths, val_err, "o-", label="validation error")
plt.xlabel("max_depth")
plt.ylabel("error")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 1c. The same choice by 5-fold cross-validation

The validation set has only 920 e-mails, so the curve above is noisy. Cross-validation uses all of `X_trainval`: it
splits it into 5 folds, fits on 4 folds and evaluates on the fifth, 5 times. For each depth, compute the mean and
the standard deviation of the error over the 5 folds with `cross_val_score(..., cv=cv5)` (it returns accuracies).

In [ ]:
cv_mean, cv_std = [], []
for depth in depths:
    # Answer 1.3: cross_val_score of a tree with this max_depth on X_trainval, cv=cv5; store mean and std of the error
    raise NotImplementedError("Answer 1.3")
cv_mean, cv_std = np.array(cv_mean), np.array(cv_std)
best_depth_cv = depths[int(np.argmin(cv_mean))]
print(f"best depth by cross-validation: {best_depth_cv}, error {cv_mean.min():.3f} +/- {cv_std[np.argmin(cv_mean)]:.3f}")

plt.figure(figsize=(7, 4))
plt.errorbar(depths, cv_mean, yerr=cv_std, fmt="o-", capsize=3, label="5-fold cross-validation error")
plt.plot(depths, val_err, "o--", alpha=0.6, label="validation error (1b)")
plt.xlabel("max_depth")
plt.ylabel("error")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 1d. Reading a tree

A tree of depth 2, drawn with the names of the variables. Read the first split: which variable, which threshold,
and which class is the majority on each side?

In [ ]:
small_tree = DecisionTreeClassifier(max_depth=2, random_state=SEED).fit(X_train, y_train)
plt.figure(figsize=(12, 5))
plot_tree(small_tree, feature_names=feature_names, class_names=["non-spam", "spam"], filled=True, fontsize=9)
plt.show()

### 1e. Two trees, two halves of the data

Fit one unpruned tree on the first half of the training set and one on the second half. Compute the fraction of
validation e-mails on which their predictions differ. The cell then prints the top of each tree.

In [ ]:
half = len(X_train) // 2
# Answer 1.4: two unpruned trees, on X_train[:half] and X_train[half:]; fraction of validation points where they disagree
raise NotImplementedError("Answer 1.4")
print(f"the two trees disagree on {disagreement:.1%} of the validation e-mails\n")
print("first half:\n" + export_text(tree_a, feature_names=feature_names, max_depth=2))
print("second half:\n" + export_text(tree_b, feature_names=feature_names, max_depth=2))

> **Question 1.1.** Why is the training error of an unpruned tree close to 0, and why does it say nothing about new
data?

**Your answer.**


> **Question 1.2.** Which depth do you choose, with which data? Why not with the test set?

**Your answer.**


> **Question 1.3.** Compare the depths chosen in 1b and 1c. Which choice do you trust more, and why?

**Your answer.**


> **Question 1.4.** Two trees learned on two halves of the same data disagree on about one e-mail out of seven. What
does this say about the variance of a tree? What could you do with many such trees?

**Your answer.**


## Part 2. Bootstrap and bagging by hand (30 min)

**Bootstrap.** A bootstrap sample of a set of $n$ observations is made of $n$ draws with replacement among the $n$
indices. Some observations are drawn several times, others are never drawn. A given observation is never drawn with
probability
$$\left(1 - \frac1n\right)^n \longrightarrow e^{-1} \approx 0.368 \quad (n \to \infty).$$
The observations that are not drawn are *out of bag* (OOB) for this sample.

**Bagging** (bootstrap aggregating). For $b = 1, \dots, B$: draw a bootstrap sample of the training set and fit an
unpruned tree $h_b$ on it. The bagged model is the mean
$$g_B(x) = \frac1B \sum_{b=1}^B h_b(x),$$
where $h_b(x)$ is the probability of spam given by tree $b$. It predicts spam when $g_B(x) > 1/2$.

In [ ]:
rng = np.random.default_rng(SEED)   # a new generator, so that this part does not depend on the cells above

### 2a. What a bootstrap sample contains

For each $n$, draw one bootstrap sample of the indices $0, \dots, n-1$ with `rng.integers(0, n, size=n)`. Compute
the fraction of indices never drawn and the largest number of times that one index is drawn (`np.bincount`). For
$n \geq 1000$ you should find a fraction between 0.36 and 0.38.

In [ ]:
for n in [10, 100, 1000, 2760]:
    # Answer 2.1: one bootstrap sample of size n; fraction of indices never drawn; largest multiplicity
    raise NotImplementedError("Answer 2.1")
    print(f"n = {n:4d}: never drawn {out_of_bag:.3f}, (1 - 1/n)^n = {(1 - 1 / n) ** n:.3f}, "
          f"e^-1 = {np.exp(-1):.3f}, largest multiplicity {largest}")

### 2b. Bagging in a few lines

Write `bagging_fit`, which returns a list of `B` unpruned trees, each fitted on a bootstrap sample of `(X, y)` (use
the same indices for `X` and `y`), and `bagging_predict_proba`, which returns $g_B$ on the rows of `X`.

In [ ]:
def bagging_fit(X, y, B, rng):
    # List of B unpruned trees, each fitted on a bootstrap sample of (X, y).
    trees = []
    n = len(X)
    # Answer 2.2: for b in range(B): draw n indices with replacement, fit DecisionTreeClassifier(random_state=SEED) on them
    raise NotImplementedError("Answer 2.2")
    return trees


def bagging_predict_proba(trees, X):
    # Mean over the trees of the probability of spam: g_B on the rows of X.
    # Answer 2.3: mean of the predict_proba(X)[:, 1] of the trees
    raise NotImplementedError("Answer 2.3")


trees = bagging_fit(X_train, y_train, 200, rng)
g_val = bagging_predict_proba(trees, X_val)
print(f"bagging of {len(trees)} trees: validation error {np.mean((g_val > 0.5) != y_val):.3f}")

**Error against $B$.** We do not refit anything. Stack the probabilities of the 200 trees on the validation set in
an array of shape (200, 920). The cumulative mean along the trees (`np.cumsum`) gives $G_1, g_2, \dots, g_{200}$ at
once. Compute the validation error `bag_val_err[B - 1]` of $g_B$ for $B = 1, \dots, 200$. You should find an error
above 10 % for $B = 1$ and between 6 and 7.5 % for $B \geq 50$.

In [ ]:
# Answer 2.4: array of tree probabilities (B x n_val), cumulative mean along the trees, error of each g_B
raise NotImplementedError("Answer 2.4")
for B in [1, 10, 50, 200]:
    print(f"B = {B:3d}: validation error {bag_val_err[B - 1]:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(trees) + 1), bag_val_err, label="bagging by hand")
plt.axhline(min(val_err), color="gray", linestyle="--", label="best single tree (Part 1b)")
plt.xscale("log")
plt.xlabel("number of trees B")
plt.ylabel("validation error")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 2c. `BaggingClassifier` and the out-of-bag error

Each training e-mail is out of bag for about 37 % of the trees. Predict it with these trees only: this is its OOB
prediction. The **OOB error** is the error of these predictions on the training set. It estimates the error on new
data and needs no validation set. With `oob_score=True`, scikit-learn computes it during the fit; `oob_score_` is an
accuracy.

Compute the validation error, the OOB error, and the mean fraction of distinct training e-mails in the bootstrap
samples (`bag.estimators_samples_` gives the indices drawn for each tree). You should find a fraction close to
$1 - e^{-1} \approx 0.632$.

In [ ]:
# BaggingClassifier draws the seed of each tree from its own random_state
bag = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=200, oob_score=True, n_jobs=N_JOBS,
                        random_state=SEED).fit(X_train, y_train)
# Answer 2.5: validation error, OOB error (1 - oob_score_), mean fraction of distinct indices per bootstrap sample
raise NotImplementedError("Answer 2.5")
print(f"validation error {bag_val_error:.3f}, OOB error {bag_oob_error:.3f}, distinct fraction {distinct:.3f}")

> **Question 2.1.** Does a bootstrap sample contain each observation exactly once? Which fraction of the observations
does it contain?

**Your answer.**


> **Question 2.2.** Each tree is about as bad as the unpruned tree of Part 1. Why is their average better? Which part
of the error decreases, the bias or the variance?

**Your answer.**


> **Question 2.3.** We average probabilities. The slides of the lecture use a majority vote of the trees. Can the two
rules give different predictions?

**Your answer.**


## Part 3. Random forest: out-of-bag error, number of trees, `max_features` (40 min)

A **random forest** is bagging of unpruned trees with one more random step: at each node, the tree draws $m$
variables among the $d$ variables and searches the best split among these $m$ variables only. In scikit-learn
$m$ is `max_features`. For classification the default is `"sqrt"`, here $m = \lfloor\sqrt{57}\rfloor = 7$. With
$m = d$ the forest is the bagging of Part 2.

Why this helps. Let $B$ predictors have the same variance $\sigma^2$ and the same correlation $\rho$ between any two
of them. The variance of their mean is
$$\rho\,\sigma^2 + \frac{1 - \rho}{B}\,\sigma^2 .$$
The second term goes to 0 when $B$ grows; the first one does not. Drawing the variables at each node makes the
trees less correlated, so $\rho$ is smaller.

### 3a. Training, OOB and validation errors

Fit `RandomForestClassifier(n_estimators=300, oob_score=True, n_jobs=N_JOBS, random_state=SEED)` on the training
set and print its training, OOB and validation errors. Then fit the same forest on `X_trainval` and print its OOB
error only. You should find OOB and validation errors between 4.5 and 5.5 %.

In [ ]:
# Answer 3.1: forest of 300 trees on the training set (three errors), then on X_trainval (OOB error only)
raise NotImplementedError("Answer 3.1")

### 3b. Number of trees

Fit one forest of 1000 trees. As in 2b, compute its validation error `forest_val_err[B - 1]` for $B = 1, \dots, 1000$
from the cumulative mean of the tree probabilities (`forest_1000.estimators_` is the list of the trees). The next
cell computes the OOB error for some values of $B$ (with `warm_start=True` the forest keeps its trees and adds new
ones) and the fit time against $B$.

In [ ]:
forest_1000 = RandomForestClassifier(n_estimators=1000, n_jobs=N_JOBS, random_state=SEED).fit(X_train, y_train)
# Answer 3.2: tree probabilities on X_val (one row per tree), cumulative mean, error for B = 1 to 1000
raise NotImplementedError("Answer 3.2")
for B in [1, 10, 50, 200, 1000]:
    print(f"B = {B:4d}: validation error {forest_val_err[B - 1]:.3f}")

In [ ]:
oob_B, oob_err = [20, 50, 100, 200, 500, 1000], []
growing = RandomForestClassifier(oob_score=True, warm_start=True, n_jobs=N_JOBS, random_state=SEED)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # with few trees, a few e-mails are in every bootstrap sample
    for B in oob_B:
        growing.set_params(n_estimators=B).fit(X_train, y_train)
        oob_err.append(1 - growing.oob_score_)

time_B, fit_time = [50, 100, 200, 400], []
for B in time_B:
    start = time.perf_counter()
    RandomForestClassifier(n_estimators=B, n_jobs=N_JOBS, random_state=SEED).fit(X_train, y_train)
    fit_time.append(time.perf_counter() - start)
print("OOB error:", {B: round(e, 3) for B, e in zip(oob_B, oob_err)})
print("fit time (s):", {B: round(t, 2) for B, t in zip(time_B, fit_time)})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(np.arange(1, 1001), forest_val_err, label="validation error")
axes[0].plot(oob_B, oob_err, "o-", label="OOB error")
axes[0].set_xscale("log")
axes[0].set_xlabel("number of trees B")
axes[0].set_ylabel("error")
axes[0].legend()
axes[1].plot(time_B, fit_time, "o-")
axes[1].set_xlabel("number of trees B")
axes[1].set_ylabel("fit time (s)")
for ax in axes:
    ax.grid(alpha=0.3)
plt.show()

### 3c. The parameter `max_features`

First write `mean_tree_correlation(forest, X)`: the mean, over all pairs of different trees, of the correlation
between their probabilities of spam on the rows of `X` (`np.corrcoef` of the array of 2b, then the entries above
the diagonal).

In [ ]:
def mean_tree_correlation(forest, X):
    # Mean correlation between the probabilities of spam of two different trees of the forest, on the rows of X.
    # Answer 3.3: array of tree probabilities, np.corrcoef, mean of the entries above the diagonal
    raise NotImplementedError("Answer 3.3")

For each $m$ below, fit a forest of 200 trees with `max_features=m` and `oob_score=True`, and name it `f`. Store
its OOB error, its validation error, its fit time and the mean correlation between its trees on the validation set.

In [ ]:
m_values = [1, 2, 4, 7, 15, 30, 57]
rows = []
for m in m_values:
    # Answer 3.4: forest f of 200 trees with max_features=m; time the fit; append a dictionary with the four numbers
    raise NotImplementedError("Answer 3.4")
    if m == 1:
        forest_m1 = f
table_m = pd.DataFrame(rows).set_index("m")
best_m = int(table_m["OOB error"].idxmin())
print(table_m.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(m_values, table_m["OOB error"], "o-", label="OOB error")
axes[0].plot(m_values, table_m["validation error"], "o-", label="validation error")
axes[0].set_ylabel("error")
axes[0].legend()
axes[1].plot(m_values, table_m["tree correlation"], "o-")
axes[1].set_ylabel("mean correlation between trees")
axes[2].plot(m_values, table_m["fit time (s)"], "o-")
axes[2].set_ylabel("fit time (s)")
for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("max_features m")
    ax.grid(alpha=0.3)
plt.show()

With $m = 1$, each node draws one variable and splits on it. How many distinct variables does one tree of this
forest use? `tree.tree_.feature` gives the variable of each node, and a negative value for a leaf.

In [ ]:
# Answer 3.5: distinct variables in the splits of the first tree of forest_m1 (leaves have a negative value)
raise NotImplementedError("Answer 3.5")
print(f"the first tree of the forest with m = 1 uses {n_distinct} distinct variables out of {X.shape[1]}")

> **Question 3.1.** The training error of the forest is close to 0 and its OOB error is not. Which one estimates the
generalisation error, and why?

**Your answer.**


> **Question 3.2.** For a forest, do you need a validation set?

**Your answer.**


> **Question 3.3.** Does the forest overfit when $B$ grows? What grows with $B$? Explain the plateau with the formula of
the variance.

**Your answer.**


> **Question 3.4.** With $m = d$, are all the trees identical? What method is the forest then?

**Your answer.**


> **Question 3.5.** With $m = 1$, does each tree use only one variable?

**Your answer.**


> **Question 3.6.** How does the correlation between trees change with $m$, and why? Compare $m = d$ (bagging) with a
small $m$. Does $m$ change the time only, or also the predictions? On these data, which $m$ gives the lowest OOB error,
and how would you choose $m$ on another dataset?

**Your answer.**


## Part 4. Gradient boosting: learning rate, number of trees, early stopping (50 min)

Gradient boosting builds the trees one after the other. Let $\ell(y, z)$ be a loss and $\gamma \in (0, 1]$ the
learning rate.

1. $G_0$ is the constant that minimises the empirical risk $\frac1n \sum_i \ell(y_i, G_0)$.
2. For $b = 1, \dots, B$: compute the pseudo-residuals $r_i = -\frac{\partial \ell}{\partial z}(y_i, G_{b-1}(x_i))$,
   fit a small regression tree $h_b$ to the pairs $(x_i, r_i)$, and set $G_b = G_{b-1} + \gamma\, h_b$.

For the squared loss $\ell(y, z) = \frac12 (y - z)^2$, the pseudo-residual is the residual $y_i - G_{b-1}(x_i)$. For
classification, $G_b(x)$ is a score and $p_b(x) = 1 / (1 + e^{-G_b(x)})$ is the probability of spam; with the
logistic loss of lecture 2 the pseudo-residual is $y_i - p_{b-1}(x_i)$. In scikit-learn, `learning_rate` is $\gamma$,
`n_estimators` is $B$ and `max_depth` is the depth of each tree $h_b$.

We follow the **validation log-loss**, the logistic loss computed on the validation set,
$-\frac1N \sum_i \big[y_i \log p(x_i) + (1 - y_i) \log(1 - p(x_i))\big]$ (`sklearn.metrics.log_loss`). Boosting
minimises this loss on the training set. It is less noisy than the 0/1 error on 920 e-mails.

### 4a. A toy example

Three points $x = 1, 2, 3$ with labels $y = 1, 3, 8$, the squared loss and $\gamma = 0.5$. Compute by hand (with
NumPy) $G_0$, the residuals $r_i = y_i - G_0$ and $G_1 = G_0 + \gamma h_1$, where $h_1$ fits the residuals exactly (a
tree of depth 2 can give a different value to each of the three points).

In [ ]:
x_toy = np.array([[1.0], [2.0], [3.0]])
y_toy = np.array([1.0, 3.0, 8.0])
gamma = 0.5
# Answer 4.1: g0, the residuals, and g1 at the three points (h1 equals the residuals)
raise NotImplementedError("Answer 4.1")
print("g0 =", g0, " residuals =", residuals, " g1 =", g1)

In [ ]:
# Check: scikit-learn with one tree of depth 2 must give the same g1
toy = GradientBoostingRegressor(n_estimators=1, learning_rate=0.5, max_depth=2, random_state=SEED).fit(x_toy, y_toy)
print("scikit-learn:", toy.predict(x_toy), " same as g1:", np.allclose(toy.predict(x_toy), g1))

After each step the residuals become $r_i - \gamma r_i = (1 - \gamma) r_i$. Check it with `staged_predict`, which
gives $G_1, g_2, \dots$ one after the other: write `largest_residual(gamma, n_trees)`, which returns the largest
absolute residual $\max_i |y_i - G_b(x_i)|$ for $b = 1, \dots, B$. The next lines print the ratio between two steps
and the number of trees needed to divide the residuals by 100.

In [ ]:
def largest_residual(gamma, n_trees):
    model = GradientBoostingRegressor(n_estimators=n_trees, learning_rate=gamma, max_depth=2, random_state=SEED)
    model.fit(x_toy, y_toy)
    # Answer 4.2: array of max_i |y_i - G_b(x_i)| for b = 1, ..., n_trees, with model.staged_predict(x_toy)
    raise NotImplementedError("Answer 4.2")


r0 = np.abs(y_toy - y_toy.mean()).max()   # largest residual of g0
for gamma in [0.5, 0.05]:
    largest = largest_residual(gamma, 300)
    n_needed = int(np.argmax(largest <= r0 / 100)) + 1
    print(f"gamma = {gamma}: ratio between two steps {largest[1] / largest[0]:.3f}, "
          f"trees needed to divide the residuals by 100: {n_needed}")

### 4b. Learning rate and number of trees on the spam data

Complete `boosting_curves`. It fits `GradientBoostingClassifier` on the training set, then returns three arrays of
length $B$: the training log-loss, the validation log-loss and the validation error of $G_1, \dots, g_B$. Use
`staged_predict_proba`, which gives the probabilities of $G_1, g_2, \dots$. It is a generator: it can be read once
only, so compute the log-loss and the error of the validation set in the same loop.

We compare $\gamma = 0.2$ and $\gamma = 0.05$ with trees of depth 2 and $B = 2000$. Each fit takes about half a
minute.

In [ ]:
def boosting_curves(gamma, depth, n_trees):
    model = GradientBoostingClassifier(max_depth=depth, learning_rate=gamma, n_estimators=n_trees, random_state=SEED)
    # Answer 4.3: fit on the training set; training log-loss of each G_b; one loop over staged_predict_proba(X_val) for the log-loss and the error
    raise NotImplementedError("Answer 4.3")
    return np.array(train_loss), np.array(val_loss), np.array(val_error)


def summary(name, curves):
    train_loss, val_loss, val_error = curves
    print(f"{name}: training log-loss at the end {train_loss[-1]:.4f} | validation log-loss: "
          f"lowest {val_loss.min():.3f} at B = {val_loss.argmin() + 1}, {val_loss[-1]:.3f} at the end | "
          f"validation error: lowest {val_error.min():.3f}, {val_error[-1]:.3f} at the end")


curves = {}
curves[0.2] = boosting_curves(0.2, depth=2, n_trees=2000)
summary("gamma = 0.2 ", curves[0.2])

In [ ]:
curves[0.05] = boosting_curves(0.05, depth=2, n_trees=2000)
summary("gamma = 0.05", curves[0.05])

In [ ]:
def plot_boosting(all_curves):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for (name, (train_loss, val_loss, val_error)), color in zip(all_curves.items(), ["C0", "C1", "C2"]):
        B = np.arange(1, len(val_loss) + 1)
        axes[0].plot(B, val_loss, color=color, label=f"{name}, validation")
        axes[0].plot(B, train_loss, color=color, linestyle=":", label=f"{name}, training")
        axes[1].plot(B, val_error, color=color, label=name)
    axes[0].set_yscale("log")
    axes[0].set_ylabel("log-loss")
    axes[1].set_ylabel("validation error")
    axes[1].set_ylim(0.03, 0.12)
    for ax in axes:
        ax.set_xscale("log")
        ax.set_xlabel("number of trees B")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    plt.show()


plot_boosting({"gamma = 0.2": curves[0.2], "gamma = 0.05": curves[0.05]})

You should see, for $\gamma = 0.2$: a training log-loss that goes towards 0, a validation log-loss with a minimum
between 0.14 and 0.17 for $B$ of a few hundred, then a clear rise. For $\gamma = 0.05$ the curves move about four
times slower, and the minimum of the validation log-loss is lower (between 0.12 and 0.14) and comes after more than
1000 trees.

### 4c. Deeper trees

The same with trees of depth 6, $\gamma = 0.2$ and $B = 500$, added to the figure.

In [ ]:
curves_deep = boosting_curves(0.2, depth=6, n_trees=500)
summary("gamma = 0.2, depth 6", curves_deep)
print(f"for comparison, depth 2 at B = 500: validation log-loss {curves[0.2][1][499]:.3f}")
plot_boosting({"gamma = 0.2": curves[0.2], "gamma = 0.05": curves[0.05], "gamma = 0.2, depth 6": curves_deep})

### 4d. Early stopping

With `n_iter_no_change`, scikit-learn puts aside a fraction `validation_fraction` of the data given to `fit`. It
stops adding trees when the loss on this internal validation set has not decreased for `n_iter_no_change`
consecutive trees. `n_estimators` is then only an upper bound, and `n_estimators_` is the number of trees kept.

For $\gamma = 0.2$ and $\gamma = 0.05$, fit `GradientBoostingClassifier(max_depth=2, learning_rate=gamma,
n_estimators=3000, validation_fraction=0.2, n_iter_no_change=20, random_state=SEED)` on the training set, and print
`n_estimators_` and the validation error.

In [ ]:
for gamma in [0.2, 0.05]:
    # Answer 4.4: boosting with early stopping; print n_estimators_ and the validation error
    raise NotImplementedError("Answer 4.4")

### 4e. Histogram gradient boosting

`HistGradientBoostingClassifier` bins each variable into at most 255 values before it searches the splits. It accepts
missing values, and it uses early stopping by default when $n > 10000$ (we switch it on here). On 2760 e-mails it
saves little or no time; on data with tens of thousands of rows it is much faster than `GradientBoostingClassifier`.
This is the version to use in your project.

In [ ]:
start = time.perf_counter()
hgb = HistGradientBoostingClassifier(early_stopping=True, max_iter=1000, learning_rate=0.1, random_state=SEED)
hgb.fit(X_train, y_train)
print(f"fit time {time.perf_counter() - start:.1f} s, {hgb.n_iter_} trees, validation error {error(hgb, X_val, y_val):.3f}")

> **Question 4.1.** In the toy example, how many more trees does $\gamma = 0.05$ need than $\gamma = 0.5$ to divide
the residuals by 100, and why?

**Your answer.**


> **Question 4.2.** On the spam data, which $\gamma$ needs more trees to reach the minimum of the validation loss? Is
the best $B$ the same for both?

**Your answer.**


> **Question 4.3.** Can adding trees make boosting overfit? Compare with the forest of Part 3.

**Your answer.**


> **Question 4.4.** Boosting overfits. Which of these actions help, and which make it worse: fewer trees, more trees,
a smaller $\gamma$ at fixed $B$, deeper trees?

**Your answer.**


> **Question 4.5.** Early stopping uses a validation set taken from the training data. Why must it not be the test set?

**Your answer.**


> **Question 4.6.** Why does boosting use shallow trees and the forest deep trees?

**Your answer.**


## Part 5. Data leakage and `Pipeline` (20 min)

We leave the spam data for a short experiment where the right answer is known. The data are pure noise: 200
observations of 5000 independent standard Gaussian variables, and labels 0 or 1 drawn independently of them. No
classifier can do better than an accuracy of 0.5 on new data.

The method: keep the 20 variables most correlated with the labels (`SelectKBest(f_classif, k=20)`), then fit a
logistic regression (lecture 2) on them.

In [ ]:
rng = np.random.default_rng(SEED)   # a new generator, so that this part does not depend on the cells above
X_noise = rng.standard_normal((200, 5000))
y_noise = rng.integers(0, 2, size=200)

### 5a. The wrong way

Select the 20 variables with all 200 observations, then estimate the accuracy of the logistic regression on these
20 variables by 5-fold cross-validation (`cv=cv5`). You should find an accuracy well above 0.5.

In [ ]:
# Answer 5.1: SelectKBest(f_classif, k=20) fitted on all the data, then cross_val_score of LogisticRegression(max_iter=1000)
raise NotImplementedError("Answer 5.1")
print(f"wrong way: accuracy {wrong.mean():.3f} +/- {wrong.std():.3f}")

### 5b. The right way

Put the selection and the classifier in one `Pipeline` with `make_pipeline`, and give the pipeline to
`cross_val_score`. In each split the selection is then fitted on the 4 training folds only. You should find an
accuracy close to 0.5.

In [ ]:
# Answer 5.2: make_pipeline(selection, logistic regression), then cross_val_score on X_noise, cv=cv5
raise NotImplementedError("Answer 5.2")
print(f"right way: accuracy {right.mean():.3f} +/- {right.std():.3f}")

> **Question 5.1.** Where did information from the validation folds enter the model in 5a?

**Your answer.**


> **Question 5.2.** You standardise each variable with the mean and the standard deviation of all the data before
splitting. Is it a leak? Does it change the predictions of a tree? Of $k$-NN?

**Your answer.**


> **Question 5.3.** Which steps of your project must go inside the pipeline?

**Your answer.**


## Part 6. Comparison, test set, checklist for the project (25 min)

### 6a. Four models, one protocol

We compare four models by 5-fold cross-validation on `X_trainval`, with the same folds `cv5` for all. Fill the
dictionary `chosen` with your choices: the depth of the tree (Part 1c), $m$ for the forest (Part 3c), the learning
rate and the depth for boosting (Part 4). Boosting uses early stopping, so you do not choose $B$. If you skipped a
part, use the values in the comment.

In [ ]:
# If a part is missing: {"tree_depth": 12, "forest_max_features": 1, "boosting_learning_rate": 0.2, "boosting_max_depth": 2}
# Answer 6.1: your choices from Parts 1c, 3c and 4
raise NotImplementedError("Answer 6.1")
print(chosen)
models = {
    "tree": DecisionTreeClassifier(max_depth=chosen["tree_depth"], random_state=SEED),
    "bagging, 100 trees": BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, n_jobs=N_JOBS,
                                            random_state=SEED),
    "forest, 500 trees": RandomForestClassifier(n_estimators=500, max_features=chosen["forest_max_features"],
                                                n_jobs=N_JOBS, random_state=SEED),
    "boosting, early stopping": GradientBoostingClassifier(max_depth=chosen["boosting_max_depth"],
                                                           learning_rate=chosen["boosting_learning_rate"],
                                                           n_estimators=3000, validation_fraction=0.2,
                                                           n_iter_no_change=20, random_state=SEED),
}

For each model, run `cross_validate(model, X_trainval, y_trainval, cv=cv5)`. It returns a dictionary with the
accuracies `test_score` of the 5 folds and the fit times `fit_time`. Append to `rows` a dictionary with the mean and
the standard deviation of the error, and the mean fit time. This cell takes about 20 seconds.

In [ ]:
rows = []
for name, model in models.items():
    # Answer 6.2: cross_validate on X_trainval with cv=cv5; append the mean and std of the error and the mean fit time
    raise NotImplementedError("Answer 6.2")
comparison = pd.DataFrame(rows).set_index("model")
comparison.round(3)

### 6b. The test set, once

Choose one model from the table. Refit it on all of `X_trainval` and compute its error on the test set. This is the
only use of the test set in the session.

In [ ]:
# Answer 6.3: name of the chosen model; refit on X_trainval; error on the test set
raise NotImplementedError("Answer 6.3")
cv_error, cv_std = comparison.loc[final_name, "CV error"], comparison.loc[final_name, "std"]
print(f"{final_name}: test error {test_error:.3f}; cross-validation error {cv_error:.3f} +/- {cv_std:.3f}")

### 6c. Forest and boosting against the number of trees

The validation curves of Part 3b (forest) and Part 4b (boosting, $\gamma = 0.2$), on the same validation set.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(forest_val_err) + 1), forest_val_err, label="random forest (3b)")
plt.plot(np.arange(1, len(curves[0.2][2]) + 1), curves[0.2][2], label="gradient boosting, gamma = 0.2, depth 2 (4b)")
plt.xscale("log")
plt.ylim(0.03, 0.15)
plt.xlabel("number of trees B")
plt.ylabel("validation error")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 6d. Checklist for your project

1. Split off the test set first. Use it once, at the end, for the model that you have chosen.
2. Choose the hyper-parameters inside the training part only: cross-validation, OOB error, early stopping.
3. Put every step that learns from the data (imputation, standardisation, selection, the model) inside a `Pipeline`.
4. Give each result as a mean and a standard deviation over the folds. A gap smaller than the standard deviation
   is not a result.
5. Start from three baselines: a constant predictor, a random forest with its OOB error, and
   `HistGradientBoostingClassifier` (or `HistGradientBoostingRegressor`) with early stopping.
6. Fix the seeds and print the versions of the libraries.

On Challenge Data, the public score of your submissions is computed on a test set. If you submit many times to tune
your model, you use this test set as a validation set, and the public score becomes optimistic.

> **Question 6.1.** Which model do you choose? Is its gap with the second model larger than the standard deviation?

**Your answer.**


> **Question 6.2.** Your model has a cross-validation error and a test error. Which one do you report, and why can the
cross-validation error of the selected model be optimistic?

**Your answer.**


> **Question 6.3.** On the figure of 6c, what happens to each method when $B$ grows? Explain with bias and variance.

**Your answer.**


> **Question 6.4.** For each method, which hyper-parameter controls overfitting?

**Your answer.**


## Going further

Each item is independent and takes 10 to 20 minutes.

1. `ExtraTreesClassifier`: compare it with the random forest (error, fit time, correlation between trees).
2. Importance of the variables: `feature_importances_` of the forest against `permutation_importance` on the
   validation set. Which variables come first?
3. AdaBoost with stumps: `AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=500)` and its
   `staged_score`, against gradient boosting.
4. Stochastic gradient boosting: `subsample=0.5` in `GradientBoostingClassifier`. Effect on the curves of 4b and on
   the time.
5. A spam filter must not lose legitimate e-mails. Compute precision, recall and the ROC curve of the forest on the
   validation set, and find a threshold such that at most 1 % of the non-spam e-mails go to spam.
6. Regression: redo Parts 3 and 4 on `fetch_california_housing` with the squared error. Note that the default of
   `RandomForestRegressor` is `max_features=1.0`, which is bagging.
7. Duplicates: the data contain duplicated e-mails. Count them, count those split between `X_trainval` and
   `X_test`, and refit the forest after removing them.
8. Pruning: `cost_complexity_pruning_path` and `ccp_alpha` chosen by cross-validation, against `max_depth`.